# Problem Statement
1. **Spectra's Objective**: The sports magazine wants to repurpose unused cricket images for social media posts under the #throwback category to boost engagement during off-seasons.

2. **Engagement Strategy**: These posts aim to drive interaction on social media by filling the content gap when there are no live updates to share.

3. **Image Analyser System**: Develop an AG2-based image analyzer that identifies the sport, analyzes team dynamics and winning probabilities, and generates engaging commentary for the images.

1. https://creadordesigns.com/wp-content/uploads/2025/03/sachine-tendulkar.webp
2. https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605


In [21]:
#!pip install Pillow

In [22]:
import os 
import warnings
warnings.filterwarnings("ignore")
import autogen
from autogen import ConversableAgent
from autogen.agentchat.contrib.multimodal_conversable_agent import MultimodalConversableAgent
from IPython.display import Markdown, display, Image

In [23]:
# load env variables
from dotenv import load_dotenv
load_dotenv('/Users/janvi/Downloads/AutoGen Handouts/.env')

True

In [24]:
# configuration for LLM
config_list_1 = {
    "config_list": [{"model": "gpt-4o", "temperature": 0.3}]
}

In [25]:
# configuration for LLM
config_list_2 = {
    "config_list": [{"model": "gpt-4o", "temperature": 0.5}]
}

In [26]:
game_analyzer_agent = MultimodalConversableAgent(
    "Game_Analyzer_Agent",
    system_message = """
You are tasked with analyzing a sports image to determine the game being played. 
Identify the type of sport (e.g., football, basketball, soccer,cricket or any other) 
from the image and recognize the teams that are involved. 
Provide basic details such as the names of players and name of the teams and the current 
state of the game (e.g., score, quarter, half-time status, overs reaminnig etc.). 
If there are any visible details that provide additional details about the game, mention them as well. 
""",
    llm_config=config_list_2,
    human_input_mode="NEVER",
)


In [27]:
probability_agent = ConversableAgent(
    "Probability_Analyzer_Agent",
    system_message = """
You are tasked with calculating the winning probability for each team based on the current score and the score they need to chase. 
Search across the web to collect information about the match and Use the game stats and their historical performences and 
calculate the probability of each team winning based on their performance up until now. Consider factors such as the score gap, time left in the game, 
and give the approximate percentage of each teams for wining the game.
""",
    llm_config= config_list_1,
    human_input_mode='NEVER',
)

In [28]:
commentory_agent = MultimodalConversableAgent(
    "Commentory_agent",
    system_message = """
You are tasked with delivering a detailed, real-time commentary on the game:
- Take help of the Game Analyzer Agent's findings (game type, score, teams, etc.).
- Use the Probability Agent's calculations to provide insight into the game's likely outcomes.
Focus on creating a seamless and detailed narrative that provides key moments, highlights, and ongoing excitement for fans.
""",
    llm_config= config_list_2,
    human_input_mode="NEVER",
)

In [29]:
user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="""
    You are assisting in analyzing sports images and coordinating among agents to extract relevant game details. If the input contains URLs, fetch and incorporate relevant details.
""",
    human_input_mode="TERMINATE",  # Try between ALWAYS, NEVER, and TERMINATE
    max_consecutive_auto_reply=0,
    code_execution_config={
        "use_docker": False  # Set to True if Docker is available
    },
)

In [30]:
group_meet = autogen.GroupChat(
    agents=[user_proxy,game_analyzer_agent, probability_agent, commentory_agent],
    messages=[],
    max_round=4,
    speaker_selection_method= "auto",
    select_speaker_auto_llm_config = config_list_1
)
group_manager = autogen.GroupChatManager(
    groupchat=group_meet,
)

In [31]:
meeting = user_proxy.initiate_chat(
    group_manager,
    message="""Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://creadordesigns.com/wp-content/uploads/2025/03/sachine-tendulkar.webp>""",
summary_method="last_msg",
)

User_proxy (to chat_manager):

Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://creadordesigns.com/wp-content/uploads/2025/03/sachine-tendulkar.webp>

--------------------------------------------------------------------------------

Next speaker: Game_Analyzer_Agent

Game_Analyzer_Agent (to chat_manager):

The image depicts a cricket player wearing the Indian national team jersey. The player is holding a cricket bat and appears to be celebrating, possibly after scoring runs. The jersey has the word "INDIA" and features the Cricket World Cup 2011 logo. There is no visible score or specific game details in the image.

--------------------------------------------------------------------------------

Next speaker: Probability_Analyzer_Agent

Probability_Analyzer_Agent (to chat_manager):

Based on the image, it seems to be related to cricket, and the player is Sachin Tendulkar, a legendary cricketer from India. Since the

In [32]:
agent_name_to_find = ["Game_Analyzer_Agent", "Probability_Analyzer_Agent", "Commentory_agent"]

display(Image(url="https://creadordesigns.com/wp-content/uploads/2025/03/sachine-tendulkar.webp", width = 600))
for agent_name in agent_name_to_find:

    # Iterate through the dictionary to find the last conversation for the specified agent
    for agent, interactions in group_manager.chat_messages.items():
        # Filter interactions by agent name
        filtered_interactions = [i for i in interactions if i.get("name") == agent_name]
        if filtered_interactions:
            # Print the last conversation's content
            # print(filtered_interactions[-1]["content"])
            display(Markdown('## ' + agent_name))
            display(Markdown(filtered_interactions[-1]["content"]))
            break
    else:
        print("Agent not found or no interactions available.")


## Game_Analyzer_Agent

The image depicts a cricket player wearing the Indian national team jersey. The player is holding a cricket bat and appears to be celebrating, possibly after scoring runs. The jersey has the word "INDIA" and features the Cricket World Cup 2011 logo. There is no visible score or specific game details in the image.

## Probability_Analyzer_Agent

Based on the image, it seems to be related to cricket, and the player is Sachin Tendulkar, a legendary cricketer from India. Since the image features the Cricket World Cup 2011 logo, it might be related to a match from that tournament. 

To calculate the winning probability for each team, we would need specific details about the match, such as the current score, the target score, the number of overs or balls remaining, and any other relevant statistics. However, since this information is not available in the image, I can provide a general approach to calculating winning probabilities in cricket:

1. **Score Gap**: Determine the difference between the current score and the target score. A smaller gap generally increases the chasing team's probability of winning.

2. **Overs/Balls Remaining**: Calculate the number of overs or balls left in the match. More remaining overs or balls typically favor the chasing team.

3. **Historical Performance**: Consider the historical performance of the teams involved. If one team has a strong track record of successful chases, their probability of winning might be higher.

4. **Current Form and Conditions**: Evaluate the current form of the players and the match conditions (e.g., pitch condition, weather). These can significantly impact the outcome.

5. **Win Prediction Models**: Use statistical models like the Duckworth-Lewis-Stern (DLS) method or other predictive algorithms to estimate winning probabilities based on the current match situation.

Without specific match details, it's challenging to provide precise winning probabilities. If you have access to live match data or specific scores, I can help you further analyze the situation.

Agent not found or no interactions available.


In [33]:
meeting1 = user_proxy.initiate_chat(
    group_manager,
    message="""Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605>.""",
summary_method="last_msg",
)

User_proxy (to chat_manager):

Analyze the sports image  and extract the game type or  teams involved or any visible players or score
    <img https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605>.

--------------------------------------------------------------------------------
Warning! Unable to load image from https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605, because cannot identify image file <_io.BytesIO object at 0x10b765760>
Warning! Unable to load image from https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605, because cannot identify image file <_io.BytesIO object at 0x10b765760>

Next speaker: Game_Analyzer_Agent

Game_Analyzer_Agent (to chat_manager):

Based on the image URL provided, it appears to be related to cricket, as it mentions "ishan-kishan" in

In [34]:
agent_name_to_find = ["Game_Analyzer_Agent", "Probability_Analyzer_Agent", "Commentory_agent"]

display(Image(url="https://c.ndtvimg.com/2025-03/30tjus8o_ishan-kishan_625x300_23_March_25.jpeg?im=FeatureCrop,algorithm=dnn,width=806,height=605", width = 600))

for agent_name in agent_name_to_find:

    # Iterate through the dictionary to find the last conversation for the specified agent
    for agent, interactions in group_manager.chat_messages.items():
        # Filter interactions by agent name
        filtered_interactions = [i for i in interactions if i.get("name") == agent_name]
        if filtered_interactions:
            # Print the last conversation's content
            # print(filtered_interactions[-1]["content"])
            display(Markdown('## ' + agent_name))
            display(Markdown(filtered_interactions[-1]["content"]))
            break
    else:
        print("Agent not found or no interactions available.")


## Game_Analyzer_Agent

Based on the image URL provided, it appears to be related to cricket, as it mentions "ishan-kishan" in the URL. Ishan Kishan is a well-known Indian cricketer. 

However, without the ability to view the image directly, I can only infer the following details:
- **Sport Type**: Cricket
- **Player Mentioned**: Ishan Kishan
- **Team Involved**: Likely to be the Indian cricket team, given Ishan Kishan's association with it.

Unfortunately, without viewing the image, I cannot provide specific details about the game state, such as the score or any other teams involved. If there are any visible scoreboards or other players in the image, those details would be essential for further analysis.

## Probability_Analyzer_Agent

To provide a more comprehensive analysis of the match and calculate the winning probability for each team, I would need additional information such as the current score, the target score, the number of overs or time left in the game, and any other relevant match statistics. 

However, based on the general context of cricket and Ishan Kishan's involvement, here are some factors that could be considered in a typical match scenario:

1. **Current Score and Target**: Knowing the current score of both teams and the target score is crucial. The closer a team is to the target with more wickets in hand, the higher their chances of winning.

2. **Overs Remaining**: The number of overs left in the game can significantly impact the probability. More overs give the batting team more opportunities to chase the target.

3. **Wickets in Hand**: The number of wickets remaining can influence the batting team's ability to chase the target. More wickets in hand generally increase the chances of winning.

4. **Historical Performance**: The historical performance of the teams and players involved can provide insight into their ability to perform under pressure.

5. **Pitch Conditions**: The condition of the pitch can affect how easy or difficult it is to score runs or take wickets.

6. **Weather Conditions**: Weather can also play a role, especially in terms of interruptions or changes in playing conditions.

If you can provide more specific details about the match, I can offer a more tailored analysis and estimate the winning probabilities for each team.

Agent not found or no interactions available.
